# CNN-EXP-024 — Strong EXP020 Backbone + Auxiliary Multi-Scale Tower

EXP023 дал важный отрицательный результат:

```text
заменить сильный EXP020 backbone на short local branch + tower
→ 32-bp AP 0.09995 → 0.06607
```

Поэтому здесь проверяется другая гипотеза:

> Multi-scale tower должен **добавлять** дальний контекст к сильному
> single-resolution backbone, а не заменять его.

Архитектура:

```text
BPE tokens
    │
    ↓
32-channel common stem
    │
    ├─────────────────────────────────────────────┐
    │                                             │
    │  STRONG BACKBONE                            │  AUXILIARY TOWER
    │  EXACTLY EXP020                             │
    │                                             │
    │  d=1,2,4,8,16,32,64,128,256,512            │  ×4   → 64 ch
    │  10 residual blocks                         │  ×16  → 96 ch
    │                                             │  ×64  → 128 ch
    │                                             │  ×256 → 192 ch
    │                                             │
    └────────────── 32-ch strong features         │
                         │                        │
                         └────── additive fusion ─┘
                                      │
                               32 token features
                                      │
                               gather back to bp
                                      │
                         heads 256/128/64/32 bp
```

Tower на каждом масштабе:

- downsample: `Conv1d(kernel=9, stride=4)`;
- два residual blocks: dilation `1` и `3`;
- `1×1` projection обратно в 32 channels;
- DirectML-safe upsample обратно к token resolution;
- additive fusion со **сохранённым сильным backbone**.

Чистота ablation:

- BPE = 200;
- target = 2048 bp;
- flank = ±6144 bp;
- input = 14336 bp;
- EXP020 backbone сохранён без изменений;
- regional heads/loss сохранены;
- batch = 4;
- seed = 42;
- 10 000 steps.

Combined theoretical token RF:

```text
strong backbone = 4093 tokens
deepest tower   = 6121 tokens
combined        = 6121 tokens
```

Главный baseline — EXP020:

```text
32-bp AP = 0.099950
AP lift  = 9.145554
TSS recall @ top 10% = 65.06%
95% TSS recall → 51.26% regions
```

EXP023 (`tower replaces backbone`) был:

```text
32-bp AP = 0.066074
```

Наша цель здесь — проверить именно:

```text
strong backbone + tower > strong backbone alone ?
```


In [1]:
from pathlib import Path
from collections import OrderedDict
import gc
import hashlib
import math
import re
import sys
import time
import warnings

from IPython.display import display
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    dml_weighted_profile_bce_with_logits,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()
print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = tuple(sorted(set(split.intervals("train")["contig"].astype(str))))
validation_contigs = tuple(sorted(set(split.intervals("validation")["contig"].astype(str))))
assert set(train_contigs).isdisjoint(validation_contigs)

profile_config = ProfileWindowConfig(
    target_length=2048,
    context_flank=130,
    positive_branch_fraction=0.5,
)

GLOBAL_FLANK = 6144
TARGET_LENGTH = profile_config.target_length
LOCAL_FLANK = profile_config.context_flank
GLOBAL_LENGTH = GLOBAL_FLANK + TARGET_LENGTH + GLOBAL_FLANK
LOCAL_LENGTH = LOCAL_FLANK + TARGET_LENGTH + LOCAL_FLANK
LOCAL_START_IN_GLOBAL = GLOBAL_FLANK - LOCAL_FLANK
LOCAL_END_IN_GLOBAL = LOCAL_START_IN_GLOBAL + LOCAL_LENGTH

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

cache_contigs = tuple(dict.fromkeys((*train_contigs, *validation_contigs)))

# assert GLOBAL_LENGTH == 14336
# assert LOCAL_LENGTH == 1284
# assert LOCAL_END_IN_GLOBAL - LOCAL_START_IN_GLOBAL == LOCAL_LENGTH

display(split.summary)
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))
print("Cache contigs:", len(cache_contigs))
print("Local / Global:", LOCAL_LENGTH, "/", GLOBAL_LENGTH)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Cache contigs: 12
Local / Global: 2308 / 14336


In [4]:
# ============================================================
# BPE-200 CONFIGURATION
# ============================================================

BPE_BASE_MERGES = 100
BPE_MERGES = 200

BPE_BASE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / f"bpe_softmask_{BPE_BASE_MERGES}.pt"
)

BPE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / f"bpe_softmask_{BPE_MERGES}.pt"
)

BUILD_BPE_IF_MISSING = True

print("Base artifact:", BPE_BASE_PATH)
print("Target artifact:", BPE_PATH)
print(f"BPE-{BPE_MERGES} exists:", BPE_PATH.is_file())

if BPE_MERGES >= 251:
    raise ValueError(
        "Этот notebook использует uint8 token ids; "
        "для >=251 merges нужен более широкий dtype."
    )


Base artifact: D:\Projects\EPIC\EPIC\artifacts\bpe_softmask_100.pt
Target artifact: D:\Projects\EPIC\EPIC\artifacts\bpe_softmask_200.pt
BPE-200 exists: True


In [5]:
# ============================================================
# ONE-TIME STEP: EXTEND BPE-100 -> BPE-200
# ============================================================
# Merge rules are learned ONLY from train contigs.
# Validation sequence is never used to select BPE rules.


def _norm_rules(history):
    return [tuple(int(v) for v in rule) for rule in history]


def _merge_tokens_cpu(
    tokens: torch.Tensor,
    left: int,
    right: int,
    new_token: int,
):
    """Apply one BPE rule exactly left-to-right on a 1D CPU tensor."""

    if tokens.device.type != "cpu":
        raise ValueError("_merge_tokens_cpu expects a CPU tensor")

    if tokens.numel() < 2:
        return tokens

    matches = (
        (tokens[:-1] == left)
        & (tokens[1:] == right)
    )

    starts = torch.nonzero(
        matches,
        as_tuple=False,
    ).flatten()

    if starts.numel() == 0:
        return tokens

    # Overlapping candidates are possible only for left == right.
    # Keep every second candidate inside each consecutive run.
    if left == right and starts.numel() > 1:
        new_run = torch.cat((
            torch.tensor(
                [True],
                dtype=torch.bool,
            ),
            starts[1:] != starts[:-1] + 1,
        ))

        run_ids = (
            torch.cumsum(
                new_run.to(torch.int64),
                dim=0,
            )
            - 1
        )

        run_starts = torch.nonzero(
            new_run,
            as_tuple=False,
        ).flatten()

        offsets = (
            torch.arange(
                starts.numel(),
                dtype=torch.int64,
            )
            - run_starts[run_ids]
        )

        starts = starts[
            offsets.remainder(2) == 0
        ]

    out = tokens.clone()
    out[starts] = new_token

    remove = torch.zeros(
        tokens.numel(),
        dtype=torch.bool,
    )
    remove[starts + 1] = True

    return out[~remove].contiguous()


def _apply_rules_cpu(
    tokens: torch.Tensor,
    rules,
):
    for left, right, new_token in rules:
        tokens = _merge_tokens_cpu(
            tokens,
            left,
            right,
            new_token,
        )
    return tokens


def build_bpe_target_from_base():

    if not BPE_BASE_PATH.is_file():
        raise FileNotFoundError(
            f"Нужен готовый BPE-{BPE_BASE_MERGES}: "
            f"{BPE_BASE_PATH}"
        )

    base = torch.load(
        BPE_BASE_PATH,
        map_location="cpu",
        weights_only=False,
    )

    rules = _norm_rules(
        base["merge_history"]
    )

    if len(rules) != BPE_BASE_MERGES:
        raise ValueError(
            f"Expected BPE-{BPE_BASE_MERGES}, "
            f"got {len(rules)} merges"
        )

    token_text = {
        int(k): str(v)
        for k, v in base["token_text"].items()
    }

    token_length = {
        int(k): int(v)
        for k, v in base["token_length"].items()
    }

    # Replay the existing BPE-100 once on TRAIN contigs.
    # These compact uint8 streams are then used to learn merges 101..200.
    streams = {}
    total_bp = 0
    started = time.perf_counter()

    for i, contig in enumerate(
        train_contigs,
        start=1,
    ):
        raw = torch.as_tensor(
            sequences[contig],
            dtype=torch.uint8,
        )

        tokens = torch.where(
            raw < 8,
            torch.remainder(raw, 4),
            torch.full_like(raw, 4),
        ).to(torch.uint8)

        tokens = _apply_rules_cpu(
            tokens,
            rules,
        ).to(torch.uint8).contiguous()

        streams[contig] = tokens
        total_bp += raw.numel()

        print(
            f"BPE-{BPE_BASE_MERGES} replay "
            f"{i:>3}/{len(train_contigs)} "
            f"{contig}: {tokens.numel():,} tokens"
        )

    PAIR_BASE = 256
    PAIR_SPACE = PAIR_BASE * PAIR_BASE

    while len(rules) < BPE_MERGES:

        counts = torch.zeros(
            PAIR_SPACE,
            dtype=torch.int64,
        )

        for tokens in streams.values():

            if tokens.numel() < 2:
                continue

            pair_code = (
                tokens[:-1].to(torch.int64)
                * PAIR_BASE
                + tokens[1:].to(torch.int64)
            )

            counts += torch.bincount(
                pair_code,
                minlength=PAIR_SPACE,
            )

        best = int(
            torch.argmax(counts)
        )

        best_count = int(
            counts[best]
        )

        left, right = divmod(
            best,
            PAIR_BASE,
        )

        new_token = 5 + len(rules)

        if best_count <= 0:
            raise RuntimeError(
                "No pair left to merge"
            )

        if new_token >= 256:
            raise RuntimeError(
                "uint8 token storage is no longer sufficient"
            )

        for contig in streams:
            streams[contig] = _merge_tokens_cpu(
                streams[contig],
                left,
                right,
                new_token,
            ).to(torch.uint8).contiguous()

        rules.append(
            (left, right, new_token)
        )

        token_text[new_token] = (
            token_text[left]
            + token_text[right]
        )

        token_length[new_token] = (
            token_length[left]
            + token_length[right]
        )

        total_tokens = sum(
            t.numel()
            for t in streams.values()
        )

        print(
            f"merge={len(rules):3d} "
            f"pair=({left},{right}) -> {new_token} "
            f"count={best_count:,} "
            f"compression={total_bp / total_tokens:.4f}x"
        )

    artifact = {
        "merge_history": rules,
        "token_text": token_text,
        "token_length": token_length,
        "vocab_size": 5 + len(rules),
        "num_merges": len(rules),
        "train_bp": int(total_bp),
        "source_artifact": str(BPE_BASE_PATH),
        "case_independent": True,
        "softmask_mode": "per-occurrence lowercase_count",
    }

    torch.save(
        artifact,
        BPE_PATH,
    )

    print()
    print("Saved:", BPE_PATH)
    print(
        "Minutes:",
        round(
            (time.perf_counter() - started) / 60,
            2,
        ),
    )


if not BPE_PATH.is_file():

    if BUILD_BPE_IF_MISSING:
        build_bpe_target_from_base()
    else:
        raise FileNotFoundError(
            BPE_PATH
        )

else:
    print(
        f"Using existing BPE-{BPE_MERGES} artifact"
    )


Using existing BPE-200 artifact


In [6]:
# ============================================================
# LOAD BPE-200 + ARTIFACT INTEGRITY
# ============================================================

bpe_artifact = torch.load(
    BPE_PATH,
    map_location="cpu",
    weights_only=False,
)

merge_history = [
    tuple(int(v) for v in rule)
    for rule in bpe_artifact["merge_history"]
]

token_text = {
    int(k): str(v)
    for k, v in bpe_artifact["token_text"].items()
}

token_length = {
    int(k): int(v)
    for k, v in bpe_artifact["token_length"].items()
}

vocab_size = int(
    bpe_artifact["vocab_size"]
)

assert len(merge_history) == BPE_MERGES
assert vocab_size == 5 + BPE_MERGES
assert set(range(vocab_size)).issubset(token_length)

# BPE-200 must be a strict continuation of the exact BPE-100 artifact.
base_artifact = torch.load(
    BPE_BASE_PATH,
    map_location="cpu",
    weights_only=False,
)

base_rules = [
    tuple(int(v) for v in rule)
    for rule in base_artifact["merge_history"]
]

assert len(base_rules) == BPE_BASE_MERGES
assert merge_history[:BPE_BASE_MERGES] == base_rules

for merge_index, (
    left,
    right,
    new_token,
) in enumerate(merge_history):

    assert new_token == 5 + merge_index

    assert (
        token_length[new_token]
        == token_length[left]
        + token_length[right]
    )

    assert (
        token_text[new_token]
        == token_text[left]
        + token_text[right]
    )

print(
    f"BPE-{BPE_MERGES} artifact integrity: OK"
)

PAD_TOKEN_ID = vocab_size
N_TOKEN_ID = 4

MAX_TOKEN_LENGTH = max(
    token_length.values()
)

if MAX_TOKEN_LENGTH >= 256:
    raise ValueError(
        "uint8 lowercase_count requires "
        "max token length < 256 bp"
    )

TOKEN_LENGTH_LOOKUP = torch.zeros(
    vocab_size,
    dtype=torch.int16,
)

for token_id, length in token_length.items():
    TOKEN_LENGTH_LOOKUP[token_id] = length

MERGE_HASH = hashlib.sha1(
    repr(merge_history).encode()
).hexdigest()[:12]

CACHE_FORMAT_VERSION = 2
CACHE_ENGINE = "torch_cpu_exact_left_to_right"

# Separate cache: BPE-100 cache can never be reused accidentally.
CACHE_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / (
        f"bpe{BPE_MERGES}_pretokenized_"
        f"v{CACHE_FORMAT_VERSION}_{MERGE_HASH}"
    )
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("BPE merges:", len(merge_history))
print("Vocab:", vocab_size)
print("Max token length:", MAX_TOKEN_LENGTH)
print("Merge hash:", MERGE_HASH)
print("Cache engine:", CACHE_ENGINE)
print("Cache:", CACHE_DIR)


BPE-200 artifact integrity: OK
BPE merges: 200
Vocab: 205
Max token length: 8
Merge hash: 03d3d5e77ea4
Cache engine: torch_cpu_exact_left_to_right
Cache: D:\Projects\EPIC\EPIC\artifacts\bpe200_pretokenized_v2_03d3d5e77ea4


In [7]:
# ============================================================
# EXACT ONE-TIME CPU PRETOKENIZATION — BPE-200
# ============================================================
# CPU is intentional here. The old full-contig DirectML path produced
# invalid no-merge caches on long contigs. Training itself remains GPU-based.

INDEX_CHUNK_TOKENS = 1_000_000


def reverse_complement_storage_codes_cpu(raw: torch.Tensor) -> torch.Tensor:
    if raw.device.type != "cpu":
        raise ValueError("reverse_complement_storage_codes_cpu expects CPU")
    x = raw.to(torch.int16)
    is_base = x < 8
    case = ((x >= 4) & (x < 8)).to(torch.int16) * 4
    rc = 3 - torch.remainder(x, 4) + case
    rc = torch.where(is_base, rc, torch.full_like(rc, 8))
    return rc.flip(0).to(torch.uint8).contiguous()


def _merge_tokens_and_lower_cpu(
    tokens: torch.Tensor,
    lower: torch.Tensor,
    left: int,
    right: int,
    new_token: int,
):
    """Exact left-to-right merge preserving per-token lowercase_count."""
    if tokens.device.type != "cpu" or lower.device.type != "cpu":
        raise ValueError("CPU tensors expected")
    if tokens.numel() != lower.numel():
        raise ValueError("tokens/lower length mismatch")
    if tokens.numel() < 2:
        return tokens, lower

    matches = (tokens[:-1] == left) & (tokens[1:] == right)
    starts = torch.nonzero(matches, as_tuple=False).flatten()
    if starts.numel() == 0:
        return tokens, lower

    if left == right and starts.numel() > 1:
        new_run = torch.cat((
            torch.tensor([True], dtype=torch.bool),
            starts[1:] != starts[:-1] + 1,
        ))
        run_ids = torch.cumsum(new_run.to(torch.int64), dim=0) - 1
        run_starts = torch.nonzero(new_run, as_tuple=False).flatten()
        offsets = torch.arange(starts.numel(), dtype=torch.int64) - run_starts[run_ids]
        starts = starts[offsets.remainder(2) == 0]

    out_tokens = tokens.clone()
    out_lower = lower.clone()
    out_tokens[starts] = new_token
    out_lower[starts] = lower[starts] + lower[starts + 1]

    remove = torch.zeros(tokens.numel(), dtype=torch.bool)
    remove[starts + 1] = True
    keep = ~remove

    return (
        out_tokens[keep].contiguous(),
        out_lower[keep].contiguous(),
    )


def apply_bpe_cpu(raw_cpu: torch.Tensor, *, reverse_complement: bool):
    raw = raw_cpu.to(dtype=torch.uint8, device="cpu").contiguous()
    if reverse_complement:
        raw = reverse_complement_storage_codes_cpu(raw)

    tokens = torch.where(
        raw < 8,
        torch.remainder(raw, 4),
        torch.full_like(raw, 4),
    ).to(torch.uint8).contiguous()
    lower = ((raw >= 4) & (raw < 8)).to(torch.uint8).contiguous()

    for left, right, new_token in merge_history:
        tokens, lower = _merge_tokens_and_lower_cpu(
            tokens, lower, left, right, new_token
        )

    return tokens, lower


def safe_name(contig: str):
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", contig)[:80]
    suffix = hashlib.sha1(contig.encode()).hexdigest()[:10]
    return f"{stem}_{suffix}"


def cache_path(contig: str, orientation: str):
    return CACHE_DIR / f"{safe_name(contig)}.{orientation}.pt"


def _build_token_ends(token_ids: torch.Tensor):
    n_tokens = token_ids.numel()
    ends = torch.empty(n_tokens, dtype=torch.int32)
    running_bp = 0

    for start in range(0, n_tokens, INDEX_CHUNK_TOKENS):
        stop = min(start + INDEX_CHUNK_TOKENS, n_tokens)
        lengths = TOKEN_LENGTH_LOOKUP[token_ids[start:stop].long()].to(torch.int32)
        chunk_ends = torch.cumsum(lengths, dim=0) + running_bp
        ends[start:stop] = chunk_ends
        running_bp = int(chunk_ends[-1])

    return ends, running_bp


def cache_is_compatible(path: Path, contig: str, orientation: str):
    if not path.is_file():
        return False
    try:
        p = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
        return (
            int(p.get("cache_format_version", -1)) == CACHE_FORMAT_VERSION
            and p.get("cache_engine") == CACHE_ENGINE
            and p.get("merge_hash") == MERGE_HASH
            and int(p.get("bpe_merges", -1)) == BPE_MERGES
            and p.get("contig") == contig
            and p.get("orientation") == orientation
            and "token_ids" in p
            and "lowercase_count" in p
            and "token_ends" in p
        )
    except Exception:
        return False


def pretokenize_contig_cpu(contig: str, orientation: str, overwrite=False):
    if orientation not in {"plus", "rc"}:
        raise ValueError(orientation)

    path = cache_path(contig, orientation)
    if not overwrite and cache_is_compatible(path, contig, orientation):
        print(f"{contig} [{orientation}] cached")
        return path

    raw = torch.as_tensor(sequences[contig], dtype=torch.uint8).contiguous()
    lowercase_bp = int(((raw >= 4) & (raw < 8)).sum())

    ids, lower = apply_bpe_cpu(
        raw,
        reverse_complement=(orientation == "rc"),
    )
    ends, reconstructed_bp = _build_token_ends(ids)

    assert reconstructed_bp == raw.numel()
    assert ids.numel() == lower.numel() == ends.numel()
    assert int(lower.to(torch.int64).sum()) == lowercase_bp
    assert int(ids.max()) < vocab_size if ids.numel() else True

    payload = {
        "cache_format_version": CACHE_FORMAT_VERSION,
        "cache_engine": CACHE_ENGINE,
        "contig": contig,
        "orientation": orientation,
        "length_bp": int(raw.numel()),
        "lowercase_bp": lowercase_bp,
        "token_ids": ids,
        "lowercase_count": lower,
        "token_ends": ends,
        "merge_hash": MERGE_HASH,
        "bpe_merges": BPE_MERGES,
        "vocab_size": vocab_size,
    }
    torch.save(payload, path)

    print(
        f"{contig} [{orientation}] bp={raw.numel():,} "
        f"tokens={ids.numel():,} compression={raw.numel() / ids.numel():.3f}x"
    )

    del raw, ids, lower, ends, payload
    gc.collect()
    return path


In [8]:
# ============================================================
# CPU-ONLY BPE SEMANTIC SMOKE TEST
# ============================================================
# Full-contig BPE is intentionally CPU-only.
# DirectML is not touched here.

smoke_contig = train_contigs[0]

smoke_raw = torch.as_tensor(
    sequences[smoke_contig][:GLOBAL_LENGTH],
    dtype=torch.uint8,
).contiguous()

if smoke_raw.numel() < GLOBAL_LENGTH:
    smoke_raw = torch.cat((
        smoke_raw,
        torch.full(
            (GLOBAL_LENGTH - smoke_raw.numel(),),
            8,
            dtype=torch.uint8,
        ),
    ))

for rc in (False, True):

    ids, lower = apply_bpe_cpu(
        smoke_raw,
        reverse_complement=rc,
    )

    lengths = TOKEN_LENGTH_LOOKUP[
        ids.long()
    ].to(torch.int32)

    assert ids.dtype == torch.uint8
    assert lower.dtype == torch.uint8
    assert ids.numel() == lower.numel()
    assert int(lengths.sum()) == GLOBAL_LENGTH

    assert torch.all(
        lower.to(torch.int16)
        <= lengths.to(torch.int16)
    )

    source = (
        reverse_complement_storage_codes_cpu(smoke_raw)
        if rc
        else smoke_raw
    )

    expected_lowercase = int(
        (
            (source >= 4)
            & (source < 8)
        ).sum()
    )

    assert (
        int(lower.to(torch.int64).sum())
        == expected_lowercase
    )

    print(
        f"{'RC' if rc else 'PLUS'}: "
        f"tokens={ids.numel():,} "
        f"compression="
        f"{GLOBAL_LENGTH / ids.numel():.3f}x"
    )

print(
    f"BPE-{BPE_MERGES} CPU semantic check: OK"
)


PLUS: tokens=4,348 compression=3.297x
RC: tokens=4,360 compression=3.288x
BPE-200 CPU semantic check: OK


In [9]:
# ============================================================
# BUILD EXACT BPE-200 PRETOKENIZED CACHE — ONE TIME, RESUME-SAFE
# ============================================================
# Old DirectML cache is intentionally ignored because CACHE_DIR changed.

OVERWRITE_CACHE = False

started = time.perf_counter()
for i, contig in enumerate(cache_contigs, 1):
    print(f"\n[{i}/{len(cache_contigs)}] {contig}")
    pretokenize_contig_cpu(contig, "plus", OVERWRITE_CACHE)
    pretokenize_contig_cpu(contig, "rc", OVERWRITE_CACHE)

print()
print("Exact cache ready. Minutes:", round((time.perf_counter() - started) / 60, 2))



[1/12] NC_064035.1
NC_064035.1 [plus] cached
NC_064035.1 [rc] cached

[2/12] NC_064036.1
NC_064036.1 [plus] cached
NC_064036.1 [rc] cached

[3/12] NC_064038.1
NC_064038.1 [plus] cached
NC_064038.1 [rc] cached

[4/12] NC_064039.1
NC_064039.1 [plus] cached
NC_064039.1 [rc] cached

[5/12] NC_064040.1
NC_064040.1 [plus] cached
NC_064040.1 [rc] cached

[6/12] NC_064043.1
NC_064043.1 [plus] cached
NC_064043.1 [rc] cached

[7/12] NC_064045.1
NC_064045.1 [plus] cached
NC_064045.1 [rc] cached

[8/12] NC_064047.1
NC_064047.1 [plus] cached
NC_064047.1 [rc] cached

[9/12] NC_064048.1
NC_064048.1 [plus] cached
NC_064048.1 [rc] cached

[10/12] NC_064034.1
NC_064034.1 [plus] cached
NC_064034.1 [rc] cached

[11/12] NC_064041.1
NC_064041.1 [plus] cached
NC_064041.1 [rc] cached

[12/12] NC_064042.1
NC_064042.1 [plus] cached
NC_064042.1 [rc] cached

Exact cache ready. Minutes: 0.0


In [10]:
# ============================================================
# FULL CACHE AUDIT
# ============================================================
# Verifies metadata, bp reconstruction, lowercase counts and boundary index.

AUDIT_CHUNK_TOKENS = 1_000_000


def audit_cache_file(contig: str, orientation: str):
    path = cache_path(contig, orientation)
    if not cache_is_compatible(path, contig, orientation):
        raise RuntimeError(f"Incompatible cache: {path}")

    p = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    ids = p["token_ids"]
    lower = p["lowercase_count"]
    ends = p["token_ends"]

    raw = torch.as_tensor(sequences[contig], dtype=torch.uint8)
    expected_bp = raw.numel()
    expected_lower = int(((raw >= 4) & (raw < 8)).sum())

    assert int(p["length_bp"]) == expected_bp
    assert ids.dtype == torch.uint8
    assert lower.dtype == torch.uint8
    assert ends.dtype == torch.int32
    assert ids.numel() == lower.numel() == ends.numel()
    assert ids.numel() > 0
    assert int(ids.max()) < vocab_size

    running_bp = 0
    lower_total = 0

    for start in range(0, ids.numel(), AUDIT_CHUNK_TOKENS):
        stop = min(start + AUDIT_CHUNK_TOKENS, ids.numel())
        chunk_ids = ids[start:stop]
        chunk_lower = lower[start:stop]
        chunk_ends = ends[start:stop]
        lengths = TOKEN_LENGTH_LOOKUP[chunk_ids.long()].to(torch.int32)
        expected_ends = torch.cumsum(lengths, dim=0) + running_bp

        assert torch.equal(chunk_ends, expected_ends)
        assert torch.all(chunk_lower.to(torch.int16) <= lengths.to(torch.int16))

        running_bp = int(expected_ends[-1])
        lower_total += int(chunk_lower.to(torch.int64).sum())

    assert running_bp == expected_bp
    assert lower_total == expected_lower == int(p["lowercase_bp"])

    return {
        "contig": contig,
        "orientation": orientation,
        "bp": expected_bp,
        "tokens": ids.numel(),
        "compression": expected_bp / ids.numel(),
        "lowercase_bp": lower_total,
    }


audit_rows = []
started = time.perf_counter()

for i, contig in enumerate(cache_contigs, 1):
    print(f"[{i}/{len(cache_contigs)}] {contig}")
    for orientation in ("plus", "rc"):
        row = audit_cache_file(contig, orientation)
        audit_rows.append(row)
        print(
            f"  {orientation:4s} tokens={row['tokens']:,} "
            f"compression={row['compression']:.3f}x"
        )

cache_audit = pd.DataFrame(audit_rows)
display(cache_audit)
print("Audit minutes:", round((time.perf_counter() - started) / 60, 2))
print("CACHE AUDIT: OK")


[1/12] NC_064035.1
  plus tokens=6,575,767 compression=3.350x
  rc   tokens=6,572,480 compression=3.352x
[2/12] NC_064036.1
  plus tokens=6,214,416 compression=3.346x
  rc   tokens=6,211,319 compression=3.348x
[3/12] NC_064038.1
  plus tokens=5,669,670 compression=3.351x
  rc   tokens=5,674,624 compression=3.348x
[4/12] NC_064039.1
  plus tokens=5,406,734 compression=3.347x
  rc   tokens=5,407,155 compression=3.347x
[5/12] NC_064040.1
  plus tokens=5,323,448 compression=3.356x
  rc   tokens=5,321,541 compression=3.357x
[6/12] NC_064043.1
  plus tokens=4,943,267 compression=3.357x
  rc   tokens=4,953,485 compression=3.350x
[7/12] NC_064045.1
  plus tokens=4,889,799 compression=3.352x
  rc   tokens=4,891,140 compression=3.351x
[8/12] NC_064047.1
  plus tokens=4,596,057 compression=3.345x
  rc   tokens=4,593,831 compression=3.346x
[9/12] NC_064048.1
  plus tokens=4,074,317 compression=3.367x
  rc   tokens=4,073,645 compression=3.367x
[10/12] NC_064034.1
  plus tokens=6,627,064 compression

,contig,orientation,bp,tokens,compression,lowercase_bp
0,NC_064035.1,plus,22031953,6575767,3.350477,5303374
1,NC_064035.1,rc,22031953,6572480,3.352152,5303374
2,NC_064036.1,plus,20793841,6214416,3.346065,5356286
3,NC_064036.1,rc,20793841,6211319,3.347734,5356286
4,NC_064038.1,plus,19000648,5669670,3.351279,7404189
5,NC_064038.1,rc,19000648,5674624,3.348354,7404189
6,NC_064039.1,plus,18096078,5406734,3.346952,4604642
7,NC_064039.1,rc,18096078,5407155,3.346691,4604642
8,NC_064040.1,plus,17865913,5323448,3.356079,5523134
9,NC_064040.1,rc,17865913,5321541,3.357282,5523134


Audit minutes: 0.02
CACHE AUDIT: OK


In [11]:
# ============================================================
# LAZY MMAP CACHE
# ============================================================
# These are mmap handles, not full copies of all tensors in RAM.
# Keeping all 24 orientation files mapped avoids repeated torch.load in training.

CACHE_MAX_ITEMS = max(32, 2 * len(cache_contigs))
_CACHE_LRU = OrderedDict()


def load_cached_lazy(contig: str, orientation: str):
    key = (contig, orientation)

    if key in _CACHE_LRU:
        item = _CACHE_LRU.pop(key)
        _CACHE_LRU[key] = item
        return item

    path = cache_path(contig, orientation)
    if not cache_is_compatible(path, contig, orientation):
        raise RuntimeError(f"Missing/incompatible cache: {path}")

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
        mmap=True,
    )

    item = {
        "payload": payload,  # keeps mmap storage alive
        "length_bp": int(payload["length_bp"]),
        "ids": payload["token_ids"],
        "lower": payload["lowercase_count"],
        "ends": payload["token_ends"],
    }

    _CACHE_LRU[key] = item
    while len(_CACHE_LRU) > CACHE_MAX_ITEMS:
        _CACHE_LRU.popitem(last=False)
        gc.collect()

    return item


print("Lazy mmap cache ready.")
print("Mapped files now:", len(_CACHE_LRU))
print("No full-contig lengths/cumsum tensors are created in RAM.")


Lazy mmap cache ready.
Mapped files now: 0
No full-contig lengths/cumsum tensors are created in RAM.


In [12]:
# ============================================================
# FAST WINDOW SLICING — NO BPE MERGES HERE
# ============================================================


def _search_token_index(ends: torch.Tensor, bp_coordinate: int):
    query = torch.tensor(bp_coordinate, dtype=ends.dtype)
    return int(torch.searchsorted(ends, query, right=True))


def slice_cached(contig, genomic_start, genomic_end, *, rc=False):
    orientation = "rc" if rc else "plus"
    item = load_cached_lazy(contig, orientation)
    L = item["length_bp"]

    if rc:
        start, end = L - genomic_end, L - genomic_start
    else:
        start, end = genomic_start, genomic_end

    if end - start != GLOBAL_LENGTH:
        raise AssertionError((start, end, GLOBAL_LENGTH))

    left_pad = max(0, -start)
    right_pad = max(0, end - L)
    a, b = max(0, start), min(L, end)

    ids_all = item["ids"]
    lower_all = item["lower"]
    ends_all = item["ends"]

    if b > a:
        first = _search_token_index(ends_all, a)
        last = _search_token_index(ends_all, b - 1)
        stop = last + 1

        core_ids = ids_all[first:stop]
        core_lower = lower_all[first:stop]
        core_lengths = TOKEN_LENGTH_LOOKUP[core_ids.long()].to(torch.int16)
        core_ends_abs = ends_all[first:stop]

        # Search only the small token slice, not the full 20M-element index.
        coords = torch.arange(a, b, dtype=core_ends_abs.dtype)
        core_map = (
            torch.searchsorted(core_ends_abs, coords, right=True)
            + left_pad
        ).to(torch.long)
    else:
        core_ids = torch.empty(0, dtype=torch.uint8)
        core_lower = torch.empty(0, dtype=torch.uint8)
        core_lengths = torch.empty(0, dtype=torch.int16)
        core_map = torch.empty(0, dtype=torch.long)

    # N is intentionally never BPE-merged, so 1 padded bp = 1 N token.
    left_ids = torch.full((left_pad,), N_TOKEN_ID, dtype=torch.uint8)
    left_lower = torch.zeros(left_pad, dtype=torch.uint8)
    left_lengths = torch.ones(left_pad, dtype=torch.int16)
    left_map = torch.arange(left_pad, dtype=torch.long)

    right_ids = torch.full((right_pad,), N_TOKEN_ID, dtype=torch.uint8)
    right_lower = torch.zeros(right_pad, dtype=torch.uint8)
    right_lengths = torch.ones(right_pad, dtype=torch.int16)
    right_map = (
        left_pad
        + core_ids.numel()
        + torch.arange(right_pad, dtype=torch.long)
    )

    ids = torch.cat((left_ids, core_ids, right_ids)).contiguous()
    lower = torch.cat((left_lower, core_lower, right_lower)).contiguous()
    lengths = torch.cat((left_lengths, core_lengths, right_lengths)).contiguous()
    bp_to_token = torch.cat((left_map, core_map, right_map)).contiguous()

    assert bp_to_token.numel() == GLOBAL_LENGTH
    assert ids.numel() == lengths.numel() == lower.numel()
    assert bp_to_token.min() >= 0
    assert bp_to_token.max() < ids.numel()

    return {
        "ids": ids,
        "lengths": lengths,
        "softmask": lower.float() / lengths.float(),
        "bp_to_token": bp_to_token,
    }


def build_cached_both_batch(tiles):
    items = []

    # Ordering intentionally matches batch.x_both_strands:
    # all PLUS rows first, then all RC rows.
    for rc in (False, True):
        for i in range(len(tiles)):
            tile = tiles.iloc[i]
            item = slice_cached(
                str(tile["contig"]),
                int(tile["target_start"]) - GLOBAL_FLANK,
                int(tile["target_end"]) + GLOBAL_FLANK,
                rc=rc,
            )
            item["local_map"] = item["bp_to_token"][
                LOCAL_START_IN_GLOBAL:LOCAL_END_IN_GLOBAL
            ]
            assert item["local_map"].numel() == LOCAL_LENGTH
            items.append(item)

    T = max(x["ids"].numel() for x in items)
    N = len(items)

    token_ids = torch.full((N, T), PAD_TOKEN_ID, dtype=torch.long)
    token_lengths = torch.zeros((N, T), dtype=torch.float32)
    softmask = torch.zeros((N, T), dtype=torch.float32)
    token_mask = torch.zeros((N, T), dtype=torch.float32)  # float for DirectML
    local_map = torch.empty((N, LOCAL_LENGTH), dtype=torch.long)

    for row, x in enumerate(items):
        n = x["ids"].numel()
        token_ids[row, :n] = x["ids"].long()
        token_lengths[row, :n] = x["lengths"].float()
        softmask[row, :n] = x["softmask"]
        token_mask[row, :n] = 1.0
        local_map[row] = x["local_map"]

    return {
        "token_ids": token_ids.contiguous(),
        "token_lengths": token_lengths.contiguous(),
        "softmask": softmask.contiguous(),
        "token_mask": token_mask.contiguous(),
        "local_map": local_map.contiguous(),
    }


# Real batch smoke-test.
smoke_batch = train_generator.sample_batch(batch_size=4)
t0 = time.perf_counter()
smoke_global = build_cached_both_batch(smoke_batch.tiles)
cache_build_ms = (time.perf_counter() - t0) * 1000

print("Cached batch build ms:", round(cache_build_ms, 2))
print("Token batch:", smoke_global["token_ids"].shape)
print("Mapped files:", len(_CACHE_LRU))
assert smoke_global["token_ids"].shape[0] == 8
assert smoke_global["local_map"].shape == (8, LOCAL_LENGTH)
print("Cached slicing smoke-test: OK")


# ============================================================
# BPE TOKEN-LENGTH vs GLOBAL CNN RF DIAGNOSTIC
# ============================================================
# Current Global encoder theoretical receptive field:
RF_LIMIT_TOKENS = 6121

probe_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=4242,
)

probe_token_counts = []

RF_PROBE_BATCHES = 8
RF_PROBE_BATCH_SIZE = 4

for _ in range(RF_PROBE_BATCHES):
    probe_batch = probe_generator.sample_batch(
        batch_size=RF_PROBE_BATCH_SIZE
    )

    probe_global = build_cached_both_batch(
        probe_batch.tiles
    )

    counts = (
        probe_global["token_mask"]
        .sum(dim=1)
        .cpu()
        .numpy()
    )

    probe_token_counts.extend(
        counts.tolist()
    )

probe_token_counts = pd.Series(
    probe_token_counts,
    dtype="float64",
)

print()
print("=== BPE TOKENS PER 14,336-bp GLOBAL WINDOW ===")
print("samples:", len(probe_token_counts))
print("min:", int(probe_token_counts.min()))
print("median:", round(float(probe_token_counts.median()), 1))
print("p90:", round(float(probe_token_counts.quantile(0.90)), 1))
print("p95:", round(float(probe_token_counts.quantile(0.95)), 1))
print("max:", int(probe_token_counts.max()))

fraction_within_rf = float(
    (probe_token_counts <= RF_LIMIT_TOKENS).mean()
)

print(
    "fraction <= RF(6121 tokens):",
    f"{fraction_within_rf:.3f}",
)

print(
    "fraction > RF(6121 tokens):",
    f"{1.0 - fraction_within_rf:.3f}",
)

if fraction_within_rf < 0.90:
    print(
        "WARNING: many windows exceed the current token RF. "
        "If AP stops improving, expand Global CNN RF before adding more context."
    )
else:
    print(
        "RF diagnostic: most sampled windows fit inside the combined multi-scale RF."
    )


Cached batch build ms: 10.51
Token batch: torch.Size([8, 4360])
Mapped files: 6
Cached slicing smoke-test: OK

=== BPE TOKENS PER 14,336-bp GLOBAL WINDOW ===
samples: 64
min: 4173
median: 4273.5
p90: 4363.7
p95: 4381.2
max: 4408
fraction <= RF(6121 tokens): 1.000
fraction > RF(6121 tokens): 0.000
RF diagnostic: most sampled windows fit inside the combined multi-scale RF.


In [13]:
# ============================================================
# MODEL — STRONG EXP020 BACKBONE + AUXILIARY MULTI-SCALE TOWER
# ============================================================

STRONG_DILATIONS = (
    1, 2, 4, 8, 16,
    32, 64, 128, 256, 512,
)

STRONG_TOKEN_RF = (
    1
    + sum(
        4 * d
        for d in STRONG_DILATIONS
    )
)

assert STRONG_TOKEN_RF == 4093

REGION_WIDTHS = (
    256,
    128,
    64,
    32,
)

assert all(
    TARGET_LENGTH % width == 0
    for width in REGION_WIDTHS
)

GLOBAL_BASE_CHANNELS = 32

TOWER_CHANNELS = (
    64,
    96,
    128,
    192,
)

TOWER_BLOCK_DILATIONS = (
    1,
    3,
)

TOWER_STRIDE = 4
TOWER_KERNEL = 9


def compute_tower_token_rf():
    """
    RF of the deepest auxiliary tower path,
    measured in original BPE-token coordinates.

    The tower starts from the common stem.
    """
    rf = 1
    jump = 1

    for _ in TOWER_CHANNELS:
        # k=9, stride=4 downsampling convolution.
        rf += (
            TOWER_KERNEL - 1
        ) * jump

        jump *= TOWER_STRIDE

        # Two residual blocks per level.
        # Each residual block has two k=3 convs.
        for dilation in TOWER_BLOCK_DILATIONS:
            rf += (
                2
                * (3 - 1)
                * dilation
                * jump
            )

    return int(rf)


TOWER_TOKEN_RF = compute_tower_token_rf()
assert TOWER_TOKEN_RF == 6121

GLOBAL_TOKEN_RF = max(
    STRONG_TOKEN_RF,
    TOWER_TOKEN_RF,
)

assert GLOBAL_TOKEN_RF == 6121
assert GLOBAL_TOKEN_RF == RF_LIMIT_TOKENS


class GlobalResidualBlock(nn.Module):
    def __init__(
        self,
        channels,
        dilation,
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            3,
            padding=dilation,
            dilation=dilation,
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            3,
            padding=dilation,
            dilation=dilation,
        )

        self.act = nn.GELU()

    def forward(
        self,
        x,
        mask,
    ):
        m = (
            mask
            .unsqueeze(1)
            .to(dtype=x.dtype)
        )

        residual = x

        x = (
            self.act(
                self.conv1(x)
            )
            * m
        )

        return (
            self.act(
                self.conv2(x)
                + residual
            )
            * m
        )


def repeat_upsample_to_base(
    x,
    scale,
    target_length,
):
    """
    DirectML-safe nearest-repeat upsampling.

    Stride-4, k=9, padding=4 gives ceil(L/4) at each level.
    Repetition therefore reaches or slightly exceeds the original
    token length; right-side excess is cropped.
    """
    B, C, L = x.shape

    y = (
        x.unsqueeze(-1)
        .expand(
            B,
            C,
            L,
            scale,
        )
        .reshape(
            B,
            C,
            L * scale,
        )
    )

    if y.shape[-1] < target_length:
        raise RuntimeError(
            (
                y.shape,
                scale,
                target_length,
            )
        )

    return (
        y[
            :,
            :,
            :target_length,
        ]
        .contiguous()
    )


class GlobalStrongBackbonePlusTower(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size + 1,
            16,
            padding_idx=PAD_TOKEN_ID,
        )

        # Exactly the EXP020 18 -> 32 projection.
        self.stem = nn.Sequential(
            nn.Conv1d(
                18,
                GLOBAL_BASE_CHANNELS,
                1,
            ),
            nn.GELU(),
        )

        # Strong branch = EXP020 backbone unchanged.
        self.strong_blocks = nn.ModuleList(
            GlobalResidualBlock(
                GLOBAL_BASE_CHANNELS,
                dilation,
            )
            for dilation
            in STRONG_DILATIONS
        )

        # Auxiliary hierarchical tower in parallel with the strong branch.
        downsamples = []
        tower_blocks = []
        tower_projections = []

        previous_channels = (
            GLOBAL_BASE_CHANNELS
        )

        for channels in TOWER_CHANNELS:
            downsamples.append(
                nn.Conv1d(
                    previous_channels,
                    channels,
                    TOWER_KERNEL,
                    stride=TOWER_STRIDE,
                    padding=TOWER_KERNEL // 2,
                )
            )

            tower_blocks.append(
                nn.ModuleList([
                    GlobalResidualBlock(
                        channels,
                        dilation,
                    )
                    for dilation
                    in TOWER_BLOCK_DILATIONS
                ])
            )

            tower_projections.append(
                nn.Conv1d(
                    channels,
                    GLOBAL_BASE_CHANNELS,
                    1,
                )
            )

            previous_channels = channels

        self.tower_downsamples = (
            nn.ModuleList(
                downsamples
            )
        )

        self.tower_blocks = (
            nn.ModuleList(
                tower_blocks
            )
        )

        self.tower_projections = (
            nn.ModuleList(
                tower_projections
            )
        )

        self.tower_act = nn.GELU()

    def forward(
        self,
        ids,
        lengths,
        softmask,
        mask,
    ):
        emb = self.embedding(ids)

        length_feature = (
            lengths.to(emb.dtype)
            / float(MAX_TOKEN_LENGTH)
        ).unsqueeze(-1)

        soft_feature = (
            softmask.to(emb.dtype)
            .unsqueeze(-1)
        )

        x = torch.cat(
            (
                emb,
                soft_feature,
                length_feature,
            ),
            dim=-1,
        )

        x = (
            x
            * mask.unsqueeze(-1).to(
                dtype=x.dtype
            )
        )

        stem = self.stem(
            x.transpose(
                1,
                2,
            ).contiguous()
        )

        base_mask = mask

        stem = (
            stem
            * base_mask.unsqueeze(1).to(
                dtype=stem.dtype
            )
        )

        # --------------------------------------------------------
        # STRONG EXP020 BRANCH
        # --------------------------------------------------------
        strong = stem

        for block in self.strong_blocks:
            strong = block(
                strong,
                base_mask,
            )

        # --------------------------------------------------------
        # AUXILIARY MULTI-SCALE TOWER
        # --------------------------------------------------------
        tower = stem
        tower_mask = base_mask

        cumulative_stride = 1
        base_length = stem.shape[-1]

        tower_sum = torch.zeros_like(
            strong
        )

        for (
            downsample,
            blocks,
            projection,
        ) in zip(
            self.tower_downsamples,
            self.tower_blocks,
            self.tower_projections,
        ):
            cumulative_stride *= (
                TOWER_STRIDE
            )

            tower = self.tower_act(
                downsample(
                    tower
                )
            )

            # Length-compatible DirectML-safe coarse mask.
            tower_mask = (
                tower_mask[
                    :,
                    ::TOWER_STRIDE,
                ]
                .contiguous()
            )

            if (
                tower.shape[-1]
                != tower_mask.shape[-1]
            ):
                raise RuntimeError(
                    (
                        tower.shape,
                        tower_mask.shape,
                    )
                )

            tower = (
                tower
                * tower_mask.unsqueeze(1).to(
                    dtype=tower.dtype
                )
            )

            for block in blocks:
                tower = block(
                    tower,
                    tower_mask,
                )

            projected = projection(
                tower
            )

            upsampled = (
                repeat_upsample_to_base(
                    projected,
                    cumulative_stride,
                    base_length,
                )
            )

            tower_sum = (
                tower_sum
                + upsampled
            )

        # The crucial EXP024 difference:
        # the strong EXP020 representation survives intact,
        # and tower features are added as auxiliary context.
        fused = (
            strong
            + tower_sum
        )

        fused = (
            fused
            * base_mask.unsqueeze(1).to(
                dtype=fused.dtype
            )
        )

        return fused


class RegionalClassifierHead(nn.Module):
    # Same EXP020 head: mean + RMS over a fixed bp region.
    def __init__(
        self,
        region_width,
    ):
        super().__init__()

        self.region_width = int(
            region_width
        )

        self.classifier = nn.Sequential(
            nn.Conv1d(
                2 * GLOBAL_BASE_CHANNELS,
                32,
                1,
            ),
            nn.GELU(),
            nn.Conv1d(
                32,
                1,
                1,
            ),
        )

    def forward(
        self,
        bp_features,
    ):
        B, C, L = bp_features.shape
        width = self.region_width

        if L % width != 0:
            raise ValueError(
                (L, width)
            )

        n_regions = L // width

        x = bp_features.reshape(
            B,
            C,
            n_regions,
            width,
        )

        mean_features = x.mean(
            dim=-1
        )

        rms_features = torch.sqrt(
            (
                x * x
            ).mean(
                dim=-1
            )
            + 1e-6
        )

        pooled = torch.cat(
            (
                mean_features,
                rms_features,
            ),
            dim=1,
        ).contiguous()

        return self.classifier(
            pooled
        )


class GlobalRegionalBPE200Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.global_encoder = (
            GlobalStrongBackbonePlusTower()
        )

        self.regional_heads = (
            nn.ModuleDict({
                str(width):
                    RegionalClassifierHead(
                        width
                    )
                for width
                in REGION_WIDTHS
            })
        )

    def forward(
        self,
        ids,
        lengths,
        softmask,
        mask,
        local_map,
    ):
        token_features = (
            self.global_encoder(
                ids,
                lengths,
                softmask,
                mask,
            )
        )

        gather_index = (
            local_map
            .unsqueeze(1)
            .expand(
                -1,
                token_features.shape[1],
                -1,
            )
            .contiguous()
        )

        bp_features = torch.gather(
            token_features,
            2,
            gather_index,
        )

        bp_features = bp_features[
            :,
            :,
            LOCAL_FLANK:-LOCAL_FLANK,
        ]

        outputs = {}

        for width in REGION_WIDTHS:
            outputs[width] = (
                self.regional_heads[
                    str(width)
                ](
                    bp_features
                )
            )

        return outputs


def create_model():
    return GlobalRegionalBPE200Model()


def forward_both(
    model,
    global_batch,
    *,
    grad=True,
):
    B2 = global_batch[
        "token_ids"
    ].shape[0]

    if B2 % 2 != 0:
        raise ValueError(
            f"Expected PLUS+RC rows, got {B2}"
        )

    B = B2 // 2

    g = {
        key: value.contiguous().to(device)
        for key, value
        in global_batch.items()
    }

    context = (
        torch.enable_grad()
        if grad
        else torch.no_grad()
    )

    with context:
        raw = model(
            g["token_ids"],
            g["token_lengths"],
            g["softmask"],
            g["token_mask"],
            g["local_map"],
        )

        outputs = {
            width: (
                logits[:B],
                logits[B:],
            )
            for width, logits
            in raw.items()
        }

    return outputs


def make_region_targets_cpu(
    exact_target,
    base_weight,
    width,
):
    if exact_target.device.type != "cpu":
        raise ValueError(
            "Expected CPU exact_target"
        )

    if base_weight.device.type != "cpu":
        raise ValueError(
            "Expected CPU base_weight"
        )

    B = exact_target.shape[0]
    n_regions = TARGET_LENGTH // width

    target = (
        exact_target
        .float()
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .amax(
            dim=-1
        )
    )

    valid = (
        (
            base_weight > 0
        )
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .all(
            dim=-1
        )
        .float()
    )

    return (
        target.contiguous(),
        valid.contiguous(),
    )


def stable_region_bce(
    logits,
    targets,
    valid,
    *,
    max_pos_weight=32.0,
):
    targets = targets.to(
        dtype=logits.dtype
    )

    valid = valid.to(
        dtype=logits.dtype
    )

    with torch.no_grad():
        pos = (
            targets * valid
        ).sum()

        neg = (
            (1.0 - targets)
            * valid
        ).sum()

        if float(
            pos.detach().cpu()
        ) > 0:
            pos_weight = (
                neg / pos
            ).clamp(
                min=1.0,
                max=max_pos_weight,
            )
        else:
            pos_weight = torch.ones(
                (),
                dtype=logits.dtype,
                device=logits.device,
            )

    element = (
        torch.clamp(
            logits,
            min=0,
        )
        - logits * targets
        + torch.log1p(
            torch.exp(
                -torch.abs(logits)
            )
        )
    )

    class_weight = (
        1.0
        + targets
        * (
            pos_weight - 1.0
        )
    )

    weighted = (
        element
        * class_weight
        * valid
    )

    denom = valid.sum().clamp(
        min=1.0
    )

    return (
        weighted.sum() / denom,
        float(
            pos_weight
            .detach()
            .cpu()
        ),
    )


model = create_model().to(device)

encoder_parameters = sum(
    p.numel()
    for p in model.global_encoder.parameters()
)

head_parameters = sum(
    p.numel()
    for p in model.regional_heads.parameters()
)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

strong_parameters = sum(
    p.numel()
    for p in model.global_encoder.strong_blocks.parameters()
)

tower_parameters = sum(
    p.numel()
    for p in (
        list(
            model.global_encoder.tower_downsamples.parameters()
        )
        + list(
            model.global_encoder.tower_blocks.parameters()
        )
        + list(
            model.global_encoder.tower_projections.parameters()
        )
    )
)

print(
    "Global architecture:",
    "strong EXP020 backbone + auxiliary multi-scale tower",
)

print(
    "Strong branch RF:",
    STRONG_TOKEN_RF,
    "tokens",
)

print(
    "Tower deepest RF:",
    TOWER_TOKEN_RF,
    "tokens",
)

print(
    "Base channels:",
    GLOBAL_BASE_CHANNELS,
)

print(
    "Tower channels:",
    TOWER_CHANNELS,
)

print(
    "Tower cumulative strides:",
    (4, 16, 64, 256),
)

print(
    "Strong-block parameters:",
    f"{strong_parameters:,}",
)

print(
    "Tower parameters:",
    f"{tower_parameters:,}",
)

print(
    "Global token RF:",
    GLOBAL_TOKEN_RF,
    "tokens",
)

print(
    "Global input:",
    GLOBAL_LENGTH,
    "bp",
)

print(
    "Region widths:",
    REGION_WIDTHS,
)

print(
    "Encoder parameters:",
    f"{encoder_parameters:,}",
)

print(
    "Regional head parameters:",
    f"{head_parameters:,}",
)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

model.eval()

smoke_outputs = forward_both(
    model,
    smoke_global,
    grad=False,
)

for width in REGION_WIDTHS:
    pp, mp = smoke_outputs[
        width
    ]

    expected_regions = (
        TARGET_LENGTH // width
    )

    print(
        f"{width:>3} bp:",
        pp.shape,
        mp.shape,
    )

    assert pp.shape == (
        4,
        1,
        expected_regions,
    )

    assert mp.shape == (
        4,
        1,
        expected_regions,
    )

print(
    "TRUE REGIONAL forward: OK"
)


Global architecture: strong EXP020 backbone + auxiliary multi-scale tower
Strong branch RF: 4093 tokens
Tower deepest RF: 6121 tokens
Base channels: 32
Tower channels: (64, 96, 128, 192)
Tower cumulative strides: (4, 16, 64, 256)
Strong-block parameters: 62,080
Tower parameters: 1,222,112
Global token RF: 6121 tokens
Global input: 14336 bp
Region widths: (256, 128, 64, 32)
Encoder parameters: 1,288,096
Regional head parameters: 8,452
Total parameters: 1,296,548
256 bp: torch.Size([4, 1, 8]) torch.Size([4, 1, 8])
128 bp: torch.Size([4, 1, 16]) torch.Size([4, 1, 16])
 64 bp: torch.Size([4, 1, 32]) torch.Size([4, 1, 32])
 32 bp: torch.Size([4, 1, 64]) torch.Size([4, 1, 64])
TRUE REGIONAL forward: OK


In [14]:
# ============================================================
# ONE REAL REGIONAL BACKWARD STEP
# ============================================================

model.train()

optimizer = DirectMLAdam(
    model.parameters(),
    lr=1e-3,
)

optimizer.zero_grad(
    set_to_none=True
)

batch = smoke_batch

plus_exact = (
    batch.plus_target
    .contiguous()
)

minus_exact = (
    batch.minus_target_reverse
    .contiguous()
)

plus_base_w = (
    batch.plus_loss_weight
    .contiguous()
)

minus_base_w = (
    batch.minus_loss_weight_reverse
    .contiguous()
)

with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    outputs = forward_both(
        model,
        smoke_global,
        grad=True,
    )

    losses = {}
    pos_weights = {}

    for width in REGION_WIDTHS:
        pp, mp = outputs[
            width
        ]

        plus_target_r, plus_valid_r = (
            make_region_targets_cpu(
                plus_exact,
                plus_base_w,
                width,
            )
        )

        minus_target_r, minus_valid_r = (
            make_region_targets_cpu(
                minus_exact,
                minus_base_w,
                width,
            )
        )

        plus_loss, plus_pw = (
            stable_region_bce(
                pp,
                plus_target_r.to(device),
                plus_valid_r.to(device),
            )
        )

        minus_loss, minus_pw = (
            stable_region_bce(
                mp,
                minus_target_r.to(device),
                minus_valid_r.to(device),
            )
        )

        losses[width] = (
            plus_loss + minus_loss
        ) / 2.0

        pos_weights[width] = (
            plus_pw + minus_pw
        ) / 2.0

    total_loss = sum(
        losses.values()
    ) / len(losses)

    total_loss.backward()

    encoder_grad = sum(
        float(
            p.grad.detach()
            .abs()
            .sum()
            .cpu()
        )
        for p
        in model.global_encoder.parameters()
        if p.grad is not None
    )

    heads_grad = sum(
        float(
            p.grad.detach()
            .abs()
            .sum()
            .cpu()
        )
        for p
        in model.regional_heads.parameters()
        if p.grad is not None
    )

    optimizer.step()

fallback = [
    str(w.message)
    for w in caught
    if "fall back"
    in str(w.message).lower()
]

for width in REGION_WIDTHS:
    print(
        f"{width:>3} bp "
        f"loss={losses[width].item():.4f} "
        f"pos_weight={pos_weights[width]:.2f}"
    )

print(
    "Total:",
    total_loss.item(),
)

print(
    "Encoder grad:",
    encoder_grad,
)

print(
    "Heads grad:",
    heads_grad,
)

print(
    "CPU fallback:",
    fallback,
)

assert encoder_grad > 0
assert heads_grad > 0
assert not fallback

print(
    "Regional backward + optimizer step: OK"
)


256 bp loss=1.2158 pos_weight=13.40
128 bp loss=1.1844 pos_weight=20.40
 64 bp loss=1.0828 pos_weight=23.83
 32 bp loss=1.0784 pos_weight=28.12
Total: 1.140356421470642
Encoder grad: 0.7807373537798412
Heads grad: 1.3401933265849948
CPU fallback: []
Regional backward + optimizer step: OK


In [15]:
# ============================================================
# 10 000-STEP BPE-200 GLOBAL REGIONAL CLASSIFIER
# ============================================================

STEPS = 10_000
BATCH_SIZE = 4
LOG_EVERY = 100
SEED = 42

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP024_BPE200_STRONG_BACKBONE_PLUS_MULTISCALE_TOWER"
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_PATH = (
    RUN_DIR
    / f"checkpoint_step_{STEPS:05d}.pt"
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

torch.manual_seed(SEED)

generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)

model = create_model().to(device)

optimizer = DirectMLAdam(
    model.parameters(),
    lr=1e-3,
)

model.train()

history = []

ema = {
    width: None
    for width
    in REGION_WIDTHS
}

ema_total = None

seen = 0
prep_seconds = 0.0

started = time.perf_counter()

with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    for step in range(
        1,
        STEPS + 1,
    ):
        t0 = time.perf_counter()

        batch = generator.sample_batch(
            batch_size=BATCH_SIZE
        )

        gb = build_cached_both_batch(
            batch.tiles
        )

        prep_seconds += (
            time.perf_counter()
            - t0
        )

        plus_exact = (
            batch.plus_target
            .contiguous()
        )

        minus_exact = (
            batch.minus_target_reverse
            .contiguous()
        )

        plus_base_w = (
            batch.plus_loss_weight
            .contiguous()
        )

        minus_base_w = (
            batch.minus_loss_weight_reverse
            .contiguous()
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = forward_both(
            model,
            gb,
            grad=True,
        )

        losses = {}
        pos_weights = {}

        for width in REGION_WIDTHS:
            pp, mp = outputs[
                width
            ]

            plus_target_r, plus_valid_r = (
                make_region_targets_cpu(
                    plus_exact,
                    plus_base_w,
                    width,
                )
            )

            minus_target_r, minus_valid_r = (
                make_region_targets_cpu(
                    minus_exact,
                    minus_base_w,
                    width,
                )
            )

            plus_loss, plus_pw = (
                stable_region_bce(
                    pp,
                    plus_target_r.to(device),
                    plus_valid_r.to(device),
                )
            )

            minus_loss, minus_pw = (
                stable_region_bce(
                    mp,
                    minus_target_r.to(device),
                    minus_valid_r.to(device),
                )
            )

            losses[width] = (
                plus_loss + minus_loss
            ) / 2.0

            pos_weights[width] = (
                plus_pw + minus_pw
            ) / 2.0

        loss = sum(
            losses.values()
        ) / len(losses)

        value = loss.item()

        if not math.isfinite(value):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        loss.backward()
        optimizer.step()

        for width in REGION_WIDTHS:
            v = losses[
                width
            ].item()

            if ema[width] is None:
                ema[width] = v
            else:
                ema[width] = (
                    0.95 * ema[width]
                    + 0.05 * v
                )

        if ema_total is None:
            ema_total = value
        else:
            ema_total = (
                0.95 * ema_total
                + 0.05 * value
            )

        seen += int(
            batch.mask.sum()
        )

        elapsed = (
            time.perf_counter()
            - started
        )

        row = {
            "step": step,
            "total_loss": value,
            "ema_total": ema_total,
            "positions_per_second": (
                seen / elapsed
            ),
            "prep_fraction": (
                prep_seconds / elapsed
            ),
        }

        for width in REGION_WIDTHS:
            row[
                f"loss_{width}"
            ] = losses[
                width
            ].item()

            row[
                f"ema_{width}"
            ] = ema[
                width
            ]

            row[
                f"pos_weight_{width}"
            ] = pos_weights[
                width
            ]

        history.append(row)

        if (
            step == 1
            or step % LOG_EVERY == 0
        ):
            head_text = " ".join(
                (
                    f"{width}:"
                    f"{losses[width].item():.4f}"
                    f"(pw={pos_weights[width]:.1f})"
                )
                for width
                in REGION_WIDTHS
            )

            print(
                f"step={step:5d} "
                f"{head_text} "
                f"total={value:.4f} "
                f"ema_total={ema_total:.4f} "
                f"positions/s="
                f"{seen / elapsed:,.0f} "
                f"prep="
                f"{100 * prep_seconds / elapsed:.1f}%"
            )

fallback = [
    str(w.message)
    for w in caught
    if "fall back"
    in str(w.message).lower()
]

if fallback:
    raise RuntimeError(
        "DirectML fallback: "
        + " | ".join(fallback)
    )

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

torch.save({
    "step": STEPS,
    "model_state_dict": {
        k: v.detach().cpu()
        for k, v
        in model.state_dict().items()
    },
    "architecture": (
        "bpe200_strong_backbone_plus_multiscale_tower_dmlsafe"
    ),
    "region_widths": (
        tuple(REGION_WIDTHS)
    ),
    "regional_pooling": (
        "mean_plus_rms"
    ),
    "bpe_merges": BPE_MERGES,
    "merge_hash": MERGE_HASH,
    "cache_format_version": (
        CACHE_FORMAT_VERSION
    ),
    "cache_engine": CACHE_ENGINE,
    "global_flank": GLOBAL_FLANK,
    "global_length": GLOBAL_LENGTH,
    "global_token_rf": GLOBAL_TOKEN_RF,
    "strong_token_rf": STRONG_TOKEN_RF,
    "tower_token_rf": TOWER_TOKEN_RF,
    "global_base_channels": GLOBAL_BASE_CHANNELS,
    "strong_dilations": tuple(
        STRONG_DILATIONS
    ),
    "tower_channels": tuple(
        TOWER_CHANNELS
    ),
    "tower_block_dilations": tuple(
        TOWER_BLOCK_DILATIONS
    ),
    "tower_stride": TOWER_STRIDE,
    "tower_kernel": TOWER_KERNEL,
    "tower_fusion": "add_to_strong_backbone",
    "tower_upsample": "nearest_repeat",
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "final_ema_by_width": {
        int(k): float(v)
        for k, v
        in ema.items()
    },
    "final_ema_total": float(
        ema_total
    ),
}, CHECKPOINT_PATH)

print(
    "Elapsed min:",
    round(
        (
            time.perf_counter()
            - started
        )
        / 60,
        2,
    ),
)

print(
    "Final EMA by width:",
    ema,
)

print(
    "Final EMA total:",
    ema_total,
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH,
)


step=    1 256:1.2177(pw=13.4) 128:1.2014(pw=20.4) 64:1.1268(pw=23.8) 32:1.0355(pw=28.1) total=1.1454 ema_total=1.1454 positions/s=21,182 prep=1.2%
step=  100 256:1.0581(pw=8.2) 128:1.1085(pw=13.3) 64:1.1364(pw=28.4) 32:1.0503(pw=32.0) total=1.0883 ema_total=1.1626 positions/s=136,494 prep=8.6%
step=  200 256:1.0704(pw=4.0) 128:1.1775(pw=6.7) 64:1.2550(pw=14.5) 32:1.3180(pw=23.0) total=1.2052 ema_total=1.1688 positions/s=142,414 prep=8.7%
step=  300 256:1.0933(pw=3.3) 128:1.0649(pw=4.5) 64:1.1401(pw=9.1) 32:1.2713(pw=16.3) total=1.1424 ema_total=1.1263 positions/s=144,690 prep=8.8%
step=  400 256:1.0049(pw=7.8) 128:1.0944(pw=9.8) 64:1.1375(pw=20.0) 32:1.0923(pw=23.8) total=1.0823 ema_total=1.1215 positions/s=145,095 prep=8.8%
step=  500 256:1.0139(pw=24.0) 128:1.0756(pw=29.0) 64:0.7673(pw=32.0) 32:0.5530(pw=32.0) total=0.8524 ema_total=1.0987 positions/s=146,409 prep=8.8%
step=  600 256:0.8722(pw=4.6) 128:0.9764(pw=10.1) 64:1.0023(pw=18.8) 32:0.9398(pw=20.4) total=0.9476 ema_total=1.05

In [16]:
# ============================================================
# FULL HOLDOUT — STRONG BACKBONE + AUXILIARY MULTI-SCALE TOWER
# ============================================================
# No sampling and no torch.linspace here.
# This is exactly the complete validation tile index.

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print("Tiles:", len(validation_tiles))
print(
    "Position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)
print(
    "Positives:",
    int(
        validation_tiles[
            "positive_positions"
        ].sum()
    ),
)
print(
    "Contigs:",
    validation_tiles[
        "contig"
    ].nunique(),
)


Tiles: 24700
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [17]:
# ============================================================
# FULL HOLDOUT VALIDATION
# REGIONAL AP + TSS COVERAGE / RECALL
# ============================================================

import numpy as np
from scipy.special import expit


TOP_REGION_FRACTIONS = (
    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.50,
)

TARGET_TSS_RECALLS = (
    0.90,
    0.95,
    0.99,
)


def exact_average_precision(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=bool,
    )

    positives = int(
        labels.sum()
    )

    if positives == 0:
        return np.nan

    order = np.argsort(
        -scores,
        kind="mergesort",
    )

    y = labels[
        order
    ].astype(
        np.int64
    )

    tp = np.cumsum(
        y,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.arange(
            1,
            y.size + 1,
            dtype=np.float64,
        )
    )

    return float(
        precision[
            y == 1
        ].sum()
        / positives
    )


def region_arrays_from_batch(
    score_plus,
    score_minus_reverse,
    mask,
    counts,
    width,
):
    B = mask.shape[0]
    n_regions = TARGET_LENGTH // width

    plus_score = score_plus[
        :,
        0,
        :,
    ]

    minus_score = score_minus_reverse[
        :,
        0,
        ::-1,
    ]

    score = np.stack(
        (
            plus_score,
            minus_score,
        ),
        axis=1,
    )

    m = mask.reshape(
        B,
        2,
        n_regions,
        width,
    )

    c = counts.reshape(
        B,
        2,
        n_regions,
        width,
    )

    valid = m.all(
        axis=-1
    )

    tss_count = (
        (
            (c > 0)
            & m
        )
        .sum(
            axis=-1
        )
        .astype(
            np.int32
        )
    )

    positive = (
        tss_count > 0
    )

    return (
        score,
        positive,
        tss_count,
        valid,
    )


def summarize_region_ranking(
    width,
    scores,
    positive,
    tss_count,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    positive = np.asarray(
        positive,
        dtype=bool,
    )

    tss_count = np.asarray(
        tss_count,
        dtype=np.int64,
    )

    n = int(scores.size)
    positive_regions = int(
        positive.sum()
    )

    total_tss = int(
        tss_count.sum()
    )

    prevalence = (
        positive_regions / n
    )

    ap = exact_average_precision(
        scores,
        positive,
    )

    ap_lift = (
        ap / prevalence
        if prevalence > 0
        else np.nan
    )

    order = np.argsort(
        -scores,
        kind="mergesort",
    )

    y_region = (
        positive[
            order
        ].astype(
            np.int64
        )
    )

    y_tss = (
        tss_count[
            order
        ].astype(
            np.int64
        )
    )

    cum_positive_regions = (
        np.cumsum(
            y_region,
            dtype=np.int64,
        )
    )

    cum_tss = (
        np.cumsum(
            y_tss,
            dtype=np.int64,
        )
    )

    ranking_rows = []

    for fraction in TOP_REGION_FRACTIONS:
        k = max(
            1,
            int(
                np.ceil(
                    fraction * n
                )
            ),
        )

        k = min(
            k,
            n,
        )

        selected_positive_regions = int(
            cum_positive_regions[
                k - 1
            ]
        )

        covered_tss = int(
            cum_tss[
                k - 1
            ]
        )

        ranking_rows.append({
            "region_width_bp": width,
            "selection_type": "top_fraction",
            "target": fraction,
            "selected_regions": k,
            "selected_fraction": (
                k / n
            ),
            "regional_precision": (
                selected_positive_regions
                / k
            ),
            "positive_region_recall": (
                selected_positive_regions
                / positive_regions
                if positive_regions > 0
                else np.nan
            ),
            "covered_tss": covered_tss,
            "tss_recall": (
                covered_tss
                / total_tss
                if total_tss > 0
                else np.nan
            ),
        })

    for recall_target in TARGET_TSS_RECALLS:
        required = (
            recall_target
            * total_tss
        )

        idx = int(
            np.searchsorted(
                cum_tss,
                required,
                side="left",
            )
        )

        idx = min(
            idx,
            n - 1,
        )

        k = idx + 1

        selected_positive_regions = int(
            cum_positive_regions[
                idx
            ]
        )

        covered_tss = int(
            cum_tss[
                idx
            ]
        )

        ranking_rows.append({
            "region_width_bp": width,
            "selection_type": "target_tss_recall",
            "target": recall_target,
            "selected_regions": k,
            "selected_fraction": (
                k / n
            ),
            "regional_precision": (
                selected_positive_regions
                / k
            ),
            "positive_region_recall": (
                selected_positive_regions
                / positive_regions
                if positive_regions > 0
                else np.nan
            ),
            "covered_tss": covered_tss,
            "tss_recall": (
                covered_tss
                / total_tss
                if total_tss > 0
                else np.nan
            ),
        })

    summary = {
        "region_width_bp": width,
        "regions": n,
        "positive_regions": positive_regions,
        "regional_prevalence": prevalence,
        "total_tss": total_tss,
        "regional_average_precision": ap,
        "regional_ap_lift": ap_lift,
    }

    return (
        summary,
        ranking_rows,
    )


def evaluate_regional_model(
    model,
    tiles,
    batch_size=8,
    progress_every=500,
):
    score_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    positive_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    tss_count_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    positions = 0
    exact_positives = 0

    started = time.perf_counter()

    was_training = model.training
    model.eval()

    try:
        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )

            for batch_no, start in enumerate(
                range(
                    0,
                    len(tiles),
                    batch_size,
                ),
                1,
            ):
                selected = tiles.iloc[
                    start:
                    start + batch_size
                ]

                batch = (
                    validation_generator
                    .make_batch(
                        selected
                    )
                )

                gb = (
                    build_cached_both_batch(
                        batch.tiles
                    )
                )

                outputs = forward_both(
                    model,
                    gb,
                    grad=False,
                )

                mask = (
                    batch.mask
                    .numpy()
                    .astype(bool)
                )

                counts = (
                    batch.count
                    .numpy()
                )

                positions += int(
                    mask.sum()
                )

                exact_positives += int(
                    (
                        mask
                        & (counts > 0)
                    ).sum()
                )

                for width in REGION_WIDTHS:
                    pp, mp = outputs[
                        width
                    ]

                    plus_score = expit(
                        pp.detach()
                        .cpu()
                        .numpy()
                    )

                    minus_score_reverse = expit(
                        mp.detach()
                        .cpu()
                        .numpy()
                    )

                    (
                        score,
                        positive,
                        tss_count,
                        valid,
                    ) = region_arrays_from_batch(
                        plus_score,
                        minus_score_reverse,
                        mask,
                        counts,
                        width,
                    )

                    score_chunks[
                        width
                    ].append(
                        score[
                            valid
                        ].astype(
                            np.float32
                        )
                    )

                    positive_chunks[
                        width
                    ].append(
                        positive[
                            valid
                        ].astype(
                            np.bool_
                        )
                    )

                    tss_count_chunks[
                        width
                    ].append(
                        tss_count[
                            valid
                        ].astype(
                            np.int16
                        )
                    )

                finished = (
                    start
                    + len(selected)
                    >= len(tiles)
                )

                if (
                    batch_no == 1
                    or batch_no % progress_every == 0
                    or finished
                ):
                    elapsed = (
                        time.perf_counter()
                        - started
                    )

                    print(
                        f"tiles="
                        f"{start + len(selected):>5}"
                        f"/{len(tiles)} "
                        f"positions="
                        f"{positions:>9} "
                        f"positions/s="
                        f"{positions / elapsed:,.0f}"
                    )

        fallback = tuple(
            str(w.message)
            for w in caught
            if "fall back"
            in str(w.message).lower()
        )

    finally:
        model.train(
            was_training
        )

    if fallback:
        raise RuntimeError(
            "DirectML fallback: "
            + " | ".join(fallback)
        )

    assert positions == int(
        tiles[
            "allowed_position_strand"
        ].sum()
    )

    assert exact_positives == int(
        tiles[
            "positive_positions"
        ].sum()
    )

    summary_rows = []
    ranking_rows = []

    for width in REGION_WIDTHS:
        scores = np.concatenate(
            score_chunks[
                width
            ]
        )

        positive = np.concatenate(
            positive_chunks[
                width
            ]
        )

        tss_count = np.concatenate(
            tss_count_chunks[
                width
            ]
        ).astype(
            np.int64
        )

        summary, rows = (
            summarize_region_ranking(
                width,
                scores,
                positive,
                tss_count,
            )
        )

        summary_rows.append(
            summary
        )

        ranking_rows.extend(
            rows
        )

    summary_df = (
        pd.DataFrame(
            summary_rows
        )
        .sort_values(
            "region_width_bp",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    ranking_df = (
        pd.DataFrame(
            ranking_rows
        )
        .sort_values(
            [
                "region_width_bp",
                "selection_type",
                "target",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    return (
        summary_df,
        ranking_df,
        elapsed,
        positions / elapsed,
    )


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert (
    checkpoint[
        "architecture"
    ]
    == "bpe200_strong_backbone_plus_multiscale_tower_dmlsafe"
)

assert tuple(
    checkpoint[
        "region_widths"
    ]
) == tuple(
    REGION_WIDTHS
)

assert (
    checkpoint[
        "regional_pooling"
    ]
    == "mean_plus_rms"
)

assert (
    checkpoint[
        "bpe_merges"
    ]
    == BPE_MERGES
)

assert (
    checkpoint[
        "merge_hash"
    ]
    == MERGE_HASH
)

assert (
    checkpoint[
        "cache_format_version"
    ]
    == CACHE_FORMAT_VERSION
)

assert (
    checkpoint[
        "global_token_rf"
    ]
    == GLOBAL_TOKEN_RF
)

validation_model = create_model().to(
    device
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

(
    regional_summary,
    regional_ranking,
    seconds,
    speed,
) = evaluate_regional_model(
    validation_model,
    validation_tiles,
    batch_size=8,
    progress_every=500,
)

SUMMARY_PATH = (
    RUN_DIR
    / f"regional_summary_{STEPS}_steps.csv"
)

RANKING_PATH = (
    RUN_DIR
    / f"regional_tss_coverage_{STEPS}_steps.csv"
)

regional_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

regional_ranking.to_csv(
    RANKING_PATH,
    index=False,
)

print()
print(
    "=== REGIONAL CLASSIFICATION SUMMARY ==="
)
display(
    regional_summary
)

print()
print(
    "=== TSS RECALL AT TOP REGION FRACTIONS ==="
)
display(
    regional_ranking[
        regional_ranking[
            "selection_type"
        ]
        == "top_fraction"
    ]
)

print()
print(
    "=== REGIONS NEEDED FOR TARGET TSS RECALL ==="
)
display(
    regional_ranking[
        regional_ranking[
            "selection_type"
        ]
        == "target_tss_recall"
    ]
)

print()
print(
    "Validation min:",
    round(
        seconds / 60,
        2,
    ),
)

print(
    "Positions/s:",
    f"{speed:,.0f}",
)

print(
    "Regional summary:",
    SUMMARY_PATH,
)

print(
    "TSS coverage:",
    RANKING_PATH,
)


tiles=    8/24700 positions=    23524 positions/s=85,912
tiles= 4000/24700 positions= 13735464 positions/s=948,128
tiles= 8000/24700 positions= 27763582 positions/s=977,749
tiles=12000/24700 positions= 41193676 positions/s=951,403
tiles=16000/24700 positions= 54896176 positions/s=959,983
tiles=20000/24700 positions= 67811662 positions/s=952,712
tiles=24000/24700 positions= 81734314 positions/s=961,729
tiles=24700/24700 positions= 83807486 positions/s=958,654

=== REGIONAL CLASSIFICATION SUMMARY ===


,region_width_bp,regions,positive_regions,regional_prevalence,total_tss,regional_average_precision,regional_ap_lift
0,256,271250,16752,0.061759,50733,0.204659,3.313860
1,128,595378,20426,0.034308,53685,0.144983,4.225978
2,64,1250248,24027,0.019218,55105,0.104632,5.444555
3,32,2560380,27982,0.010929,55730,0.074982,6.860926



=== TSS RECALL AT TOP REGION FRACTIONS ===


,region_width_bp,selection_type,target,selected_regions,selected_fraction,regional_precision,positive_region_recall,covered_tss,tss_recall
3,256,top_fraction,0.01,2713,0.010002,0.444158,0.071932,9371,0.184712
4,256,top_fraction,0.02,5425,0.020000,0.359078,0.116285,13258,0.261329
5,256,top_fraction,0.05,13563,0.050002,0.266018,0.215377,20464,0.403367
6,256,top_fraction,0.10,27125,0.100000,0.209180,0.338706,27146,0.535076
7,256,top_fraction,0.20,54250,0.200000,0.158028,0.511760,35059,0.691049
8,256,top_fraction,0.50,135625,0.500000,0.101154,0.818947,45933,0.905387
12,128,top_fraction,0.01,5954,0.010000,0.301142,0.087780,10614,0.197709
13,128,top_fraction,0.02,11908,0.020001,0.238747,0.139185,15079,0.280879
14,128,top_fraction,0.05,29769,0.050000,0.172293,0.251102,22626,0.421459
15,128,top_fraction,0.10,59538,0.100000,0.133293,0.388524,30363,0.565577



=== REGIONS NEEDED FOR TARGET TSS RECALL ===


,region_width_bp,selection_type,target,selected_regions,selected_fraction,regional_precision,positive_region_recall,covered_tss,tss_recall
0,256,target_tss_recall,0.90,132580,0.488774,0.102361,0.810112,45660,0.900006
1,256,target_tss_recall,0.95,171487,0.632210,0.087709,0.897863,48197,0.950013
2,256,target_tss_recall,0.99,229850,0.847373,0.071216,0.977137,50226,0.990007
9,128,target_tss_recall,0.90,264814,0.444783,0.063414,0.822138,48317,0.900009
10,128,target_tss_recall,0.95,355887,0.597750,0.051963,0.905366,51008,0.950135
11,128,target_tss_recall,0.99,491925,0.826240,0.040638,0.978704,53149,0.990016
18,64,target_tss_recall,0.90,516018,0.412733,0.038859,0.834561,49595,0.900009
19,64,target_tss_recall,0.95,712377,0.569789,0.030859,0.914929,52350,0.950005
20,64,target_tss_recall,0.99,1000455,0.800205,0.023556,0.980855,54554,0.990001
27,32,target_tss_recall,0.90,1017177,0.397276,0.023436,0.851941,50158,0.900018



Validation min: 1.48
Positions/s: 946,579
Regional summary: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP024_BPE200_STRONG_BACKBONE_PLUS_MULTISCALE_TOWER\run_001\regional_summary_10000_steps.csv
TSS coverage: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP024_BPE200_STRONG_BACKBONE_PLUS_MULTISCALE_TOWER\run_001\regional_tss_coverage_10000_steps.csv
